In [0]:
%pip install reportlab

In [0]:
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)

In [0]:
summary = (
    spark.table("workspace.default.gold_census_summary")
    .first()
)

age_rows = (
    spark.table("workspace.default.gold_age_breakdown")
    .orderBy("age_group")
    .collect()
)

In [0]:
summary_data = [
    ["Measure", "Count"],
    ["Households", summary.households],
    ["Persons", summary.persons],
    ["Minors", summary.minors],
    ["Seniors", summary.seniors],
]

age_data = [["Age", "Persons"]]

for row in age_rows:
    age_data.append([row.age_group, row.person_count])

In [0]:
pdf_path = "/Volumes/workspace/default/parish_data/Parish_Census_Report.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=letter,
    title="Annual Parish Census Report",
    leftMargin=0.75 * inch,
    rightMargin=0.75 * inch,
    topMargin=0.75 * inch,
    bottomMargin=0.75 * inch
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "ReportTitle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=20,
    leading=24,
    alignment=TA_CENTER,
    spaceAfter=6
)

subtitle_style = ParagraphStyle(
    "ReportSubtitle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=9,
    leading=12,
    alignment=TA_CENTER,
    textColor=colors.grey,
    spaceAfter=18
)

heading_style = ParagraphStyle(
    "ReportHeading",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=13,
    leading=16,
    spaceBefore=8,
    spaceAfter=8
)

body_style = ParagraphStyle(
    "ReportBody",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=10,
    leading=14,
    textColor=colors.black,
    spaceAfter=10
)

table_style = TableStyle([
    # header
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8E8E8")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),

    # body
    ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
    ("FONTSIZE", (0, 0), (-1, -1), 10),

    # borders
    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#B5B5B5")),

    # spacing
    ("LEFTPADDING", (0, 0), (-1, -1), 8),
    ("RIGHTPADDING", (0, 0), (-1, -1), 8),
    ("TOPPADDING", (0, 0), (-1, -1), 6),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 6),

    # alignment
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("ALIGN", (1, 1), (-1, -1), "RIGHT"),
])

summary_table = Table(
    summary_data,
    colWidths=[
        doc.width * 0.72,
        doc.width * 0.28
    ]
)

summary_table.setStyle(table_style)


age_table = Table(
    age_data,
    colWidths=[
        doc.width * 0.72,
        doc.width * 0.28
    ],
    repeatRows=1
)

age_table.setStyle(table_style)

elements = [

    Paragraph("Annual Parish Census Report", title_style),

    Paragraph(
        "Current household and demographic summary",
        subtitle_style
    ),

    Paragraph("Census Summary", heading_style),

    Paragraph(
        "The current census contains all households and persons "
        "submitted through the parish census workbook.",
        body_style
    ),

    summary_table,

    Spacer(1, 24),

    Paragraph("Age Breakdown", heading_style),

    Paragraph(
        "Ages are calculated from date of birth as of the reporting date.",
        body_style
    ),

    age_table,

    Spacer(1, 24),

    Paragraph("Notes", heading_style),

    Paragraph(
        "Records failing Silver-layer validation are not included "
        "in the published census report.",
        body_style
    ),

]
doc.build(elements)